# Zadoščanje omejitvam

Do sedaj smo se na predavanjih prepričali, da preiskovalni algoritmi lahko rešujejo podan problem, če ga lahko predstavimo kot prostor stanj, ki ustrezajo možnim (delnim) rešitvam problema. To lahko storimo tudi pri igranju iger za dva igralca, kjer prostor stanj uredimo tako, da ustreza možnim pozicijam v igri. Na tem predavanju bomo obravnavali še en razred problemov, ki jih lahko rešujemo s posebno vrsto preiskovanja, ki ji rečemo zadoščanje omejitvam.

Primer takega problema je [Sudoku](https://sl.wikipedia.org/wiki/Sudoku). Gre za nalogo zapolnjevanja praznih polj mreže (običajne) velikosti $9 \times 9$ s števili od 1 do 9. Pri tem moramo upoštevati tri omejitve:

  * v vsaki vrstici se vsako število pojavi natanko enkrat;
  * v vsakem stolpcu se vsako število pojavi natanko enkrat;
  * v vsakem bloku (pod-kvadratu polj) velikosti $3 \times 3$ se vsako število pojavi natanko enkrat.

Na prvi pogled se zdi Sudoku problem rešljiv s preiskovanjem. Množica stanj ustreza vsem možnim konfiguracijam mreže $9 \times 9$, kjer je vsako izmed polj lahko prazno ali vsebuje eno od devetih možnih števil. Začetno stanje je delno izpolnjena mreža, ki ima nekaj praznih polj (ker je število možnih začetnih stanj zelo veliko, je Sudoku zelo popularna enigmatična uganka). Končno stanje ustreza tabeli, ki nima praznih polj in zadošča trem omejitvam navedenih zgoraj. V vsakem stanju lahko v enem izmed praznih polj mreže vpišemo število od 1 do 9. 

Toda pozoren vpogled v ravnokar definiran prostor stanj pokaže, da je reševanje s preiskovanjem težavno zaradi časovne in prostorske zahtevnosti. Premisli, na primer, kakšna je stopnja razvejanja zgoraj definiranega prostora stanj za reševanje ugank Sudoku.

Pri reševanju problema Sudoku s preiskovalnimi algoritmi bi zanemarili njegovi pomembni lastnosti, ki omogočata (učinkovito) reševanje.

  * **Rešitev je neodvisna od vrstnega reda izbranih korakov** Ne pozabimo, da je pri klasičnem preiskovanju rešitev pot, kjer je vrstni red prehodov (korakov) med stanji bistven, saj je pot _zaporedje_ (in ne _množica_) prehodov. Pri uganki Sudoku je namreč vseeno ali najprej izpolnimo zgornjo levo ali spodnjo desno celico. Končna rešitev bo v obeh primerih enaka. Zato bi preiskovalno drevo po nepotrebnem vključevalo veliko različnih poti, ki ustrezajo isti rešitvi.

  * **Kršitev omejitve takoj razveljavi celotno vejo preiskovalnega drevesa** Če v nekem stanju ugotovimo, da se neko število pojavi dvakrat v vrstici (ali stolpcu, ali bloku), delna rešitev, ki ustreza temu stanju, ne more več voditi do veljavne končne rešitve. Še več, nobena razširitev te delne dodelitve ne more biti rešitev. To pomeni, da lahko zavržemo celotno poddrevo, ki nastane z razvejanjem tega vozlišča.

Klasični preiskovalni algoritmi, ki smo jih spoznali do sedaj, ne morejo sistematično izkoriščati zgornjih dveh lastnosti problema Sudoku. Zato se ga bomo lotili z novim preiskovalnim pristopom, ki mu rečemo zadoščanje omejitvam. Predavanje povzema vsebino petega poglavja z naslovom _Constraint Satisfaction Problems_ iz učbenika {cite}`russell2019aima`.

## Spremenljivke, domene, omejitve in dodelitve

Sudoku lahko formalno opišemo kot problem zadoščanja omejitvam, tako, da definiramo spremenljivke, domene njihovih vrednosti in omejitve, ki določajo dovoljene (tudi dopustne) kombinacije vrednosti spremenljivk. Spremenljivk je 81, vsaka ustreza eni celici v omrežju $9 \times 9$. Vse spremenljivke imajo isto domeno devetih (9) možnih vrednosti od 1 do 9. Omejitve pa določajo, da so dopustne vrednosti spremenljivk take, da zagotavljajo različnost števil v vsaki vrstici, vsakem stolpcu in vsakem bloku velikosti $3 \times 3$. Vsaka vrstica, vsak stolpec in vsak blok določa eno od 27 omejitev. Naloga ni več najti pot v preiskovalnem prostoru, temveč

> Poiskati dodelitev vrednosti vsem spremenljivkam tako, da je zadoščeno vsem podanim omejitvam.

Poglejmo zdaj formalne definicije osnovnih pojmov.

```{prf:definition} Problem zadoščanja omejitvam
:label: def-zadoscanje-omejitvam

Problem zadoščanja omejitvam je trojica $\left< \mathcal{X}, \mathcal{D}, \mathcal{C} \right>$, kjer je:
  - $\mathcal{X} = \{X_1, X_2, \dots, X_n\}$ končna množica _spremenljivk_;
  - $\mathcal{D} = \{D_1, D_2, \dots, D_n\}$ množica _domen spremenljivk_ iz $\mathcal{X}$, za katero velja $|\mathcal{D}| = |\mathcal{X}|$, pri čemer je domena $D_i$ množica možnih vrednosti spremenljivke $X_i$;
  - $\mathcal{C}$ množica omejitev, ko določajo dopustne kombinacije vrednosti spremenljivk.
```

Definirajmo še omejitve.

```{prf:definition} Omejitev
:label: def-omejitev

Omejitev za podan problem zadoščanja omejitvam $\left< \mathcal{X}, \mathcal{D}, \mathcal{C} \right>$ je urejen par $\left<S, R\right>$, kjer je:
  - $S = \{X_{s_1}, X_{s_2}, \dots X_{s_r}\}$ množica spremenljivk na katere se nanaša omejitev, $S \subseteq \mathcal{X}$;
  - $R \subseteq D_{s_1} \times D_{s_2} \times \dots \times D_{s_r}$ je relacija, ki določa nabor _dopustnih_ kombinacij vrednosti spremenljivk iz $S$.

Moč množice $S$ imenujemo _mestnost_ omejitve.
```

Rešitev problema zadoščanja omejitvam je dodelitev vrednosti spremenljivkam.

```{prf:definition} Dodelitev in vrste dodelitev
:label: def-dodelitev

_Dodelitev_ vrednosti spremenljivkam za podan problem zadoščanja omejitvam $\left< \mathcal{X}, \mathcal{D}, \mathcal{C} \right>$ je $d$-terica $\left<X_{a_1} = v_{a_1}, X_{a_2} = v_{a_2}, \dots, X_{a_d} = v_{a_d}\right>$, kjer $X_{a_i} \in \mathcal{X}$ in $v_{a_i} \in D_{a_i}$.

_Konsistentna_ dodelitev spremenljivkam dodeli kombinacijo vrednosti, ki ne krši nobene omejitve iz $\mathcal{C}$. Dodelitev krši omejitev, če spremenljivkam dodeli kombinacijo vrednosti, ki so izven nabora dopustnih kombinacij določenih z omejitvijo.

_Popolna_ dodelitev je $n$-terica $\left<X_1 = v_1, X_2 = v_2, \dots, X_n = v_n\right>$, kjer $v_i \in D_i$, ki dodeli vrednosti **vsem** spremenljivkam iz $\mathcal{X}$. _Delna_ dodelitev je vsaka dodelitev, ki ni popolna, in zatorej dodeli vrednost le pravi podmnožici spremenljivk iz $\mathcal{X}$.

Rešitev podanega problema zadoščanja omejitvam je _popolna_ in _konsistentna_ dodelitev vrednosti spremenljivkam.
```

Poglejmo zdaj še en primer zadoščanja omejitvam: barvanje zemljevida sedmih avstralskih regij, ki jih prikazuje slika {ref}`fig-avstralske-regije`, kjer sta sosednji regiji (regiji s skupno mejo) vedno pobarvani z različnima barvama.

```{figure} ../materiali/zemljevid-avstralskih-regij.png
---
name: fig-avstralske-regije
---
Zemljevid avstralskih regij.
```

Problem zadoščanja omejitvam definiramo tako, da je:
  * $\mathcal{X} = \{ \text{WA}, \text{NT}, \text{Q}, \text{NSW}, \text{V}, \text{SA}, \text{T} \}$, kjer vsaka spremenljivka ustreza eni avstralski regiji;
  * $D_i = \{ \text{rdeča}, \text{zelena}, \text{modra} \}$ določa množico treh možnih barv, ki so enake za vse regije iz $\mathcal{X}$:
  * $\mathcal{C} = \{$
      $\left< \{\text{WA}, \text{NT}\}, \text{WA} \neq \text{NT} \right>$,
      $\left< \{\text{WA}, \text{SA}\}, \text{WA} \neq \text{SA} \right>$,
      $\left< \{\text{NT}, \text{SA}\}, \text{NT} \neq \text{SA} \right>$,
      $\left< \{\text{NT}, \text{Q}\}, \text{NT} \neq \text{Q} \right>$,
      $\left< \{\text{Q}, \text{NSW}\}, \text{Q} \neq \text{NSW} \right>$,
      $\left< \{\text{Q}, \text{SA}\}, \text{Q} \neq \text{SA} \right>$,
      $\left< \{\text{NSW}, \text{V}\}, \text{NSW} \neq \text{V} \right>$,
      $\left< \{\text{NSW}, \text{SA}\}, \text{NSW} \neq \text{SA} \right>$,
      $\left< \{\text{V}, \text{SA}\}, \text{V} \neq \text{SA} \right>$
    $\}$ določajo omejitve pri barvanju, ki ustrezajo posameznim parom sosednjih regij na zemljevidu.

Ker so vse omejitve v tem primeru binarne (vključujejo dve spremenljivki, torej je njihova mestnost enaka 2), lahko jih ponazorimo z grafom omejitev, kot to ponazarja Slika {ref}`fig-graf-omejitev-ar`. Vozlišče v tem grafu ustrezajo spremenljivkam, povezave pa parom spremenljivk, ki nastopajo v isti omejitvi.

```{figure} ../materiali/graf-omejitev-ar.png
---
name: fig-graf-omejitev-ar
---
Graf omejitev, ki ustreza problemu barvanja zemljevida avstralskih regij iz slike {ref}`fig-avstralske-regije`.
```

Premisli kako bi zasnovali formalno predstavitev problema Sudoku. Koliko spremenljivk je v formalni predstavitvi in čemu ustrezajo, kakšne so njihove domene in katere so binarne omejitve, ki določijo nabor dopustnih rešitev. Koliko binarnih omejitev vsebuje formalna predstavitev?

## Preiskovanje s sestopanjem za reševanje problemov zadoščanja omejitvam

Preiskovalni algoritem SESTOPANJE iz {ref}`alg-pzo-sestopanje` gradi rešitev postopoma z razširjanjem delne dodelitve, ki je na začetku delovanja algoritma prazna. Algoritem najprej preveri, ali je dodelitev že popolna (vrstici 12–13). Če je popolna (vse spremenljivke imajo dodeljene vrednosti) in konsistentna z vsemi omejitvami, je algoritem našel rešitev podanega problema zadoščanja omejitvam. V nasprotnem primeru algoritem izbere eno spremenljivko brez dodeljene vrednosti (vrstica 15) in nato sistematično preizkusi vse vrednosti iz njene domene (vrstica 17). Naivna verzija algoritma izbira spremenljivke v vrstnem redu, kot ga določa urejenost `PZO.X`, vrednosti iz domene spremenljivke pa v vrstnem redu urejenosti domene `PZO.D[X]`. Delovanje hevristik MRV za spreminjanje vrstnega reda spremenljivk (funkcija `IZBERI_SPREMENLJIVKO_BREZ_DODELITVE`) in LCV za preurejanje domene vrednosti spremenljivke (funkcija `UREDI_DOMENO`) bomo pojasnili v naslednjem razdelku.

```{code-block} text
:caption: Preiskovanje s sestopanjem za podan problem zadoščanja omejitvam (PZO)
:name: alg-pzo-sestopanje
:linenos:

INPUT:
  PZO = (X, D, C)            # spremenljivke, domene, omejitve
  dodelitev                  # delna dodelitev (na začetku je prazna)
  uporabi_MRV, uporabi_LCV   # uporaba hevristik
  sklepanje                  # NONE | FC | AC3

OUTPUT:
  dodelitev                 # popolna in konsistentna dodelitev ali NEUSPEH

FUNCTION SESTOPANJE(dodelitev, PZO):

  if POPOLNA(dodelitev, PZO) then
      return dodelitev

  X ← IZBERI_SPREMENLJIVKO_BREZ_DODELITVE(dodelitev, PZO, uporabi_MRV)

  for v in UREDI_DOMENO(X, dodelitev, PZO, uporabi_LCV) do
      if KONSISTENTNA(X = v, dodelitev, PZO) then
          dodaj (X = v) v dodelitev

          kopija_D ← KOPIRAJ_DOMENE(PZO.D)              # za sestopanje

          if sklepanje ≠ NONE then
              ok ← SKLEPAJ(PZO, X, v, sklepanje)        # zmanjšaj domene
          else
              ok ← True

          if ok then
              rezultat ← SESTOPANJE(dodelitev, PZO)
              if rezultat ≠ NEUSPEH then
                  return rezultat

          rezultat ← ODSTRANI(X, rezultat)
          PZO.D ← kopija_D                            # resetiraj domene

  return NEUSPEH


FUNCTION IZBERI_SPREMENLJIVKO_BREZ_DODELITVE(dodelitev, PZO, uporabi_MRV):
  if not uporabi_MRV then
      return PRVO_SPREMENLJIVKO_BREZ_DODELITVE(PZO.X, dodelitev)
  else
      return argmin_{X brez dodelitve} |PZO.D[X]|     # hevristika MRV: spremenljivka z najmanjšo domeno


FUNCTION UREDI_DOMENO(X, dodelitev, PZO, uporabi_LCV):
  if not uprobai_LCV then
      return PZO.D[X]                                 # vrni domeno brez dodatnega urejanja
  else
      return vrednosti v iz PZO.D[X] urejene po naraščajoči
                vrednosti ST_KONFLIKTOV(X=v, dodelitev, PZO)   # hevristika LCV


FUNCTION SKLEPAJ(PZO, X, v, sklepanje):
  if sklepanje = FC then
      return FC(PZO, X, v)
  if sklepanje = AC3 then
      return AC3(PZO)                                 # ali AC3 z inicialno vrsto povezav
```

Za vsako vrednost iz domene izbrane spremenljivke algoritem najprej preveri _lokalno_ skladnost s trenutno dodelitvijo (vrstica 18, klic funkcije `KONSISTENTNA`). Lokalna skladnost pomeni, da nova dodelitev `X=v` ne krši nobene omejitve, ki vključuje spremenljivko `X` in spremenljivke v trenutni dodelitvi (preverimo torej veljavnost zgolj tistih omejitev, ki so v tem trenutku preverljive). Če ni ugotovljene kršitve omejitev, v dodelitev doda `X=v` (vrstica 19) in rekurzivno nadaljuje razširjanje trenutne dodelitve proti rešitvi (vrstice 28–31). Vmes (v vrsticah 23-26) uporabi mehanizme sklepanja za razširjanje omejitev, ki poskušata vnaprej ugotoviti težave z veljavnostjo omejitev tako, da zmanjšata domene možnih vrednosti spremenljivk. Dva, pogosto uporabljena mehanizma sklepanja FC in AC2 bom obravnavali pozneje, v ustreznem razdelku.

Če poznejši rekurzivni klic algoritma `SESTOPANJE` vrne neuspeh, algoritem prekine trenutno vejo, odstrani `X=v` iz trenutne dodelitve (vrstici 33 in 34) in poskusi naslednjo vrednost `v` spremenljivke `X`. Tako algoritem sistematično preišče prostor delnih dodelitev, pri čemer se ob prvem protislovju sestopi en korak nazaj. Če algoritem ne najde nobene dopustne dodelitve `X=v` vrne informacijo o neuspehu (vrstica 36).

### Hevristiki MRV in LCV

Osnovni preiskovalni algoritem s sestopanjem lahko bistveno pospešimo z dodatnimi mehanizmi, ki zmanjšujejo faktor razvejanja in s tem tudi velikost ter globino preiskovalnega drevesa. Brez hevristik algoritem preprosto izbere prvo spremenljivko brez dodeljene vrednosti in ji zaporedno preizkuša dodeliti vse vrednosti iz njene domene. Tak pristop ne izkorišča strukture problema in pogosto vodi do nepotrebno velikega števila neuspešnih vej.

Hevristika MRV (angl. _Minimum Remaining Values_, slov. _najmanj preostalih vrednosti_), uporabljena pri izbiri naslednje spremenljivke (vrstica 43), spremeni to strategijo tako, da najprej izbere najbolj omejeno spremenljivko — to je tisto, ki ima najmanj preostalih dopustnih vrednosti v svoji domeni. Intuicija je preprosta: če je neka spremenljivka že močno omejena, je verjetnost, da vodi do protislovja (konflikta z omejitvami), večja. Če protislovje obstaja, ga bomo tako odkrili čim prej, torej bližje korenu preiskovalnega drevesa.

S tem MRV deluje po načelu _najprej najtežji del problema_. Če bi namreč najprej obravnavali manj omejene spremenljivke, bi lahko zgradili globoko vejo delne dodelitve, šele pozneje pa ugotovili, da najbolj omejena spremenljivka sploh nima veljavne vrednosti. MRV torej povečuje verjetnost zgodnjega odkrivanja kršitev omejitev in s tem zmanjšuje število vozlišč, ki jih moramo obravnavati. Posledično se iskalno drevo praviloma bistveno zmanjša, čeprav globina preiskovalnega drevesa (število spremenljivk) ostane enaka.

Hevristika LCV (angl. _Least Constraining Value_, slov. _najmanj omejujoča vrednost_) ureja vrednosti domene izbrane spremenljivke tako, da najprej preizkusimo tisto vrednost, ki najmanj omejuje možnosti za nadaljnje dodeljevanje. Natančneje, za vsako kandidatno vrednost $v \in D_i$ spremenljivke $X_i$ ocenimo, koliko vrednosti bi zaradi te izbire izgubile domene sosednjih spremenljivk (to so spremenljivke, s katerimi $X_i$ nastopa v omejitvah), ki še nimajo dodeljenih vrednosti. Vrednost, ki povzroči najmanj izločitev (torej ohrani največ možnosti za prihodnje korake), obravnavamo najprej (vrstici 50 in 51).

Intuicija je naslednja: če neka vrednost za spremenljivko močno zoži domene njenih sosedov, obstaja večja verjetnost, da bomo kasneje naleteli na protislovje (konflikt z veljavnostjo omejitev) in morali sestopiti nazaj. Če pa izberemo vrednost, ki pusti drugim spremenljivkam čim več _svobode_, zmanjšamo tveganje zgodnjega _zastoja_ v iskanju. Hevristika LCV torej ne zmanjšuje razvejitvenega faktorja neposredno (kot MRV), temveč poskuša ohraniti fleksibilnost prihodnjih odločitev in tako zmanjšati število (prihodnjih) neuspešnih vej v iskalnem drevesu.


### Mehanizmi sklepanje za razširjanje omejitev

Mehanizmi sklepanja za razširjanje omejitev (vrstice 28–31) dodatno zmanjšujejo prostor iskanja z zmanjševanjem domen možnih vrednosti spremenljivk. Oglejmo si dva taka mehanizma, preverjanje vnaprej in povezavno konsistentnost.

Mehanizem _preverjanja vnaprej_ (angl. _forward checking_) nadgradi osnovno sestopanje tako, da omejitve ne preverja zgolj pasivno, temveč jih začne uporabljati za sprotno zmanjševanje domen spremenljivk, ki še nimajo dodeljenih vrednosti. Ko spremenljivki $X_i$ dodelimo vrednost $v \in D_i$, takoj pregledamo vse spremenljivke, ki si delijo omejitev s spremenljivko $X_i$ (tem spremenljivkam rečemo sosedi $X_i$), in iz njihovih domen odstranimo vse vrednosti, za katere $X_i = v$ povzroči neveljavnost omejitve.

```{code-block} text
:caption: Mehanizem preverjanja vnaprej (angl. _forward checking, FC_)
:name: alg-pzo-fc
:linenos:

INPUT:
  PZO, X, v

OUTPUT:
  True, če ni protislovja; sicer False

FUNCTION FC(PZO, X, v):

  for sosed Y od X (X in Y sta v isti omejitvi) do
      if Y nima dodeljene vrednosti then
          odstrani iz PZO.D[Y] vse vrednosti w, ki kršijo veljavnost omejitve ob X=v in Y=w
          if PZO.D[Y] postane prazen then
              return False
  return True
```

Ključna prednost tega mehanizma je, da morebitno protislovje odkrijemo takoj, ko se pojavi. Če katerikoli sosednji spremenljivki zaradi nove dodelitve domena postane prazna, vemo, da trenutna veja ne more voditi do rešitve, zato takoj sestopimo nazaj. Protislovje torej zaznamo takoj, že v trenutku njegovega nastanka, namesto da bi čakali, da algoritem sestopanja to ugotovi šele pozneje. Mehanizem preverjanja vnaprej tako zmanjšuje globino nepotrebnega preiskovanja in pomembno zmanjša število vozlišč v preiskovalnem drevesu. Pomembno je še poudariti, da ta mehanizem preverja le neposredne posledice trenutne dodelitve. Ne zagotavlja pa, da so omejitve med spremenljivkami brez dodeljenih vrednosti med seboj skladne.

Mehanizem sklepanja AC-3 (angl. _Arc Consistency 3_) naredi še korak dlje. Namesto da omejitve preverja le med novo dodeljeno spremenljivko $X_i$ in njenimi neposrednimi sosedi, sistematično zagotavlja tako imenovano _povezavno konsistentnost_ med vsemi pari spremenljivk, povezanimi z omejitvami.

```{code-block} text
:caption: Mehanizem povezavne konsistentnosti (angl. _Arc Consistency 3_)
:name: alg-pzo-ac3
:linenos:

INPUT:
  PZO = (X, D, C)

OUTPUT:
  True, če uspe; False, če neka domena postane prazna

FUNCTION AC3(PZO):

  vrsta ← vse povezave (Xi, Xj), kjer Xi and Xj nastopata v isti omejitvi

  while vrsta ni prazna do
      (Xi, Xj) ← POP(vrsta)
      if SPREMENI(PZO, Xi, Xj) then
          if PZO.D[Xi] je prazna then
              return False
          for sosed Xk od Xi (Xi in Xk sta v isti omejitvi) tako, da velja Xk ≠ Xj do
              PUSH(vrsta, (Xk, Xi))
  return True


FUNCTION SPREMENI(PZO, Xi, Xj):

  sprememba ← False
  for x in PZO.D[Xi] do
      if ni vrednosti y iz PZO.D[Xj], ki bi zadoščala omejitvam za Xi=x in Xj=y then
          odstrani x iz PZO.D[Xi]
          sprememba ← True
  return sprememba
```

Omejitev med spremenljivkama $X_i$ in $X_j$ je povezavno konsistentna, če za vsako vrednost $v_1 \in D_i$ obstaja vsaj ena vrednost $v_2 \in D_j$, ki je z njo skladna (torej zadovolji omejitve, ki vključujejo $X_i$ in $X_j$). Če take vrednosti ni, lahko $v_1$ odstranimo iz domene $D_i$. AC-3 to preverjanje izvaja iterativno: vsaka sprememba domene lahko povzroči nove neskladnosti, zato je treba ponovno preveriti povezane pare spremenljivk. Rezultat je močnejša oblika razširjanja omejitev kot pri preverjanju vnaprej. AC-3 lahko odkrije protislovja, še preden sploh poskusimo dodeliti novo vrednost. S tem se lahko velik del iskalnega prostora odpravi že vnaprej. Cena za to je dodatno računanje, vendar je v številnih praktičnih primerih (npr. Sudoku) ta strošek več kot poplačan z zadostnim (običajno velikim) zmanjšanjem velikosti preiskovalnega drevesa.

# Naloga za bonus točke

  1. (5 točk, rok oddaje 30. marec) Implementiraj algoritem preiskovanja s sestopanjem za reševanje problemov zadoščanja omejitvam in ga uporabi za reševanje ugank Sudoku na mreži dimenzij $4 \times 4$. Za to uganko veljajo enaka pravila kot za Sudoku $9 \times 9$, s tem, da so možne vrednosti polj od 1 do 4, in da so štirje manjši bloki $2 \times 2$, kjer se vrednosti ne smejo ponavljati.<br/><br/>Za različne primere ugank Sudoku $4 \times 4$ primerjaj in poročaj število rekurzivnih klicev navadnega sestopanja ter sestopanja z uporabo različnih hevristik in mehanizmov sklepanja.

<br/><br/>